RADIUS sta per **Remote Authentication Dial-In User Service**. Il nome deriva dall'epoca in cui gli utenti si collegavano ad internet tramite modem e linee telefoniche, il suo utilizzo si è poi esteso ad altri tipi di accesso alla rete. 

Il problema da risolvere è la gestione degli utenti in una rete con molti punti di accesso: immaginiamo un operatore che ha molti dispositivi attraverso cui gli utenti possono connettersi --> gestire separatamente su ogni dispositivo password, autorizzazioni e configurazioni è complicato, ogni modifica dovrebbe essere replicata su tutti i dispositivi.

RADIUS permette di **centralizzare le decisioni di accesso**, mantenendo distribuiti i dispositivi che collegano concretamente gli utenti alla rete.  
I partecipanti sono:
- **Utente finale** == chi vuole accedere alla rete, presenta le proprie credenziali al dispositivo di accesso.
- **NAS** (Network Access Server) == gestisce l'accesso dell'utente e applica la decisione del server
- **Server RADIUS** == riceve le richieste di accesso dal NAS, verifica le credenziali e stabilisce quali servizi autorizzare

La cosa importante da capire è che **il client che comunica con il server RADIUS non è l'utente finale, ma il NAS**! A sua volta il NAS comunica con l'utente finale però tramite il protocollo previsto dal sistema di accesso (es. PPP), mentre con il server RADIUS comunica tramite il protocollo RADIUS.

In generale il funzionamento è il seguente:
1. L'utente presenta al NAS le informazioni necessarie per autenticarsi
2. il NAS le inserisce in una richiesta RADIuS e le invia al server RADIUS
3. il server RADIUS verifica le credenziali e risponde al NAS con un messaggio di accettazione o rifiuto
4. il NAS applica la decisione e informa l'utente

RADIUS è un protocollo che fornisce le funzionalità **AAA**: **Authentication** (l'utente è davvero chi dice di essere? es. verifica credenziali), **Authorization** (quali servizi può utilizzare? es. controlla le autorizzazioni) e **Accounting** (quali risorse ha utilizzato? es. durata sessione, traffico, info per la fatturazione).

RADIUS classico è protocollo client server basato su UDP/IP, con porta 1812 per l'autenticazione. La centralizzazione è logica: possono chiaramente esserci server secondari/repliche.

La richiesta NAS prende il nome di **Access-Request**. Le risposte principali sono *Access-Accept* (accesso autorizzato), *Access-Reject* (accesso negato) e Access-Challenge (richiesta di ulteriori informazioni).

Access-Challenge non indica però necessariamente challenge crittografica: infatti alcuni metodi richiedono più messaggi, come quelli che trasportano uno scambio TLS (come vedremo).

In generale un server RADIUS può inoltrare la richiesta ad un altro server ancora, comportandosi quindi come **proxy**.  
Ciò è utile ad es. nel roaming: l'utente accede attraverso la rete ospitante, ma le sue credenziali devono essere verificate dal server del suo provider.  
Per i nostri scopi è sufficiente capire che il NAS può ricevere una decisione ottenuta da più server, concentreremo l'analisi di sicurezza però sul collegamento diretto tra NAS e server RADIUS.

Il server dispone di informazioni relative a **utenti** (credenziali, metodo di autenticazione, autorizzazioni...), **client RADIUS** (NAS autorizzati a comunicare col server), **accounting** (info sull'utilizzo dei servizi)

### Funzionamento RADIUS
Due meccanismi principali per il RADIUS originario:
- **Autenticazione delle risposte del server**, mediante segreto condiviso
- **Protezione della password dell'utente**, mediante meccanismo specifico di RADIUS

Prima di studiarli, distinguiamo i due segreti:
$$P_U = \text{password dell'utente} \quad S = \text{segreto condiviso tra NAS e server RADIUS}$$
La password permette di autenticare l'utente, mentre il segreto $S$ serve a proteggere la comunicazione tra NAS e server RADIUS. **L'utente non deve conoscere il segreto $S$**.

Tra i problemi del protocollo RADIUS originario:
- le risposte del server sono autenticate, mentre dalla parte del NAS (gli Access-Request) no
- l'autenticazione della risposta usa una costruzione basata direttamente su MD5 invece che su HMAC-MD5
- MD5 è fissato dal protocollo, quindi non è possibile sostituirlo con un algoritmo più sicuro
- lo stesso segreto viene impiegato anche per proteggere la password

Nel modello che considereremo si assume che **il NAS e il server RADIUS siano fidati**, che **l'utente possa essere attaccante** e **possa intercettare e modificare i messaggi tra NAS e server RADIUS**.  
Il fatto che NAS sia assunto fidato è necessario perché è il dispositivo che poi applica la decisione --> se fosse compromesso potrebbe consentire l'accesso a chiunque.

La preoccupazione che ci interessa riguarda le risposte del server: se il server risponde con *Access-Reject* e un attaccante riesce a trasformare il messaggio in *Access-Accept* allora è nu burdel.  
Inoltre le risposte del server contengono attributi che specificano le autorizzazioni dell'utente, quindi non basta proteggere il "sì" o il "no", ma anche quali servizi vengono concessi.

### Autenticazione della risposta RADIUS
Siano:
$$I = \text{Identifier della richiesta}, \quad A = \text{Request-Authenticator}$$
Allora il NAS genera $A$ come valore casuale a 16 byte e lo inserisce nell'Access-Request.  
In un Access-Request, $A$ **svolge il ruolo di nonce** (il nome confonde, da solo questo campo non è MAC della richiesta).  
Il NAS conserva localmente le info necessarie ad associare la richiesta alla sua risposta, incluso il valore $A$. 

Quando il server riceve la richiesta, prende la decisione e calcola un tag:
$$t = MD5(\text{Code} || I || \text{Length} || A || \text{Attributes} || S)$$
I campi Code, Identifier, Length e Attributes sono quelli della risposta. $A$ è invece il **Request-Authenticator** della richiesta originale.

Il tag viene inserito nel campo Authenticator della risposta, che in questo caso prende il nome di **Response-Authenticator**. Quindi il campo Authenticator assume due significati nel protocollo RADIUS in base al pacchetto:
- se pacchetto Access-Request --> valore casuale usato come nonce
- se pacchetto Access-Accept/Reject/Challenge --> tag di autenticazione della risposta

Quando il NAS riceve la risposta:
1. individua la richiesta corrispondente e recupera il nonce $A$ conservato
2. ricalcola il tag $t$ usando la risposta ricevuta e il segreto $S$
3. confronta il risultato con il Response-Authenticator

Se i valori coincidono --> verifica riuscita

**Perché serve il segreto?** Per impedire all'attaccante di costruire liberamente un tag valido per una risposta modificata **(autenticazione del messaggio --> protezione da falsificazioni e modifiche i.e. integrità)**
**Perché serve il nonce?** Per collegare la risposta alla richiesta corrente. Senza il nonce l'attaccante potrebbe ripresentare una vecchia risposta positiva **(nonce e gestione dello stato --> protezione da replay)**

**Perché è il NAS a scegliere il nonce?** Perché è il NAS a dover verificare la risposta, quindi è lui che deve poterla associare alla richiesta corrente. Se fosse il server a scegliere il nonce --> il NAS dovrebbe disporre di un ulteriore meccanismo per stabilire se la risposta è nuova. La presenza di molti NAS rende questo meccanismo difficile. 

Il limite di questa protezione è che sì, copre la risposta e il nonce della richiesta, **ma non copre tutti gli attributi della richiesta Access-Request**: questa distinzione sarà il punto dell'attacco a CHAP (una risposta autentica e fresca può essere la risposta a una richiesta che un attaccante ha modificato!)

#### Il formato del pacchetto
Un pacchetto RADIUS ha un'intestazione da 20 byte, seguita da un payload "Attributes" varibaile (max totale 4096 byte):

<img src="imgs/radius_header.png" width="400"/>

Tra i codici troviamo *1 = Access-Request*, *2 = Access-Accept*, *3 = Access-Reject*, *4 = Accounting-Request*, *5 = Accounting-Response*, *11 = Access-Challenge*. Ovviamente qui non serve sapere a memoria i numeri degli attributi, basta sapere come è fatto il pacchetto. 

L'identifier ha solo 8 bit quindi 256 valori possibili, serve ad **associare la risposta alla richiesta**. Da non confondere con il nonce che invece è il Request-Authenticator, è un campo da 128 bit e serve come detto a fornire freschezza crittografica (del resto con 128 bit per il paradosso del compleanno come visto le collisioni diventano probabili dopo circa 2^64 richieste!).

Gli attributi seguono il formato **Type-Length-Value* (TLV):
- Type identifica l'attributo specifico (es. User-Name, User-Password, NAS-Port, Service-Type, CHAP-Challenge... 8 bit e quindi 256 tipi possibili)
- Length ne indica la lunghezza 
- Value contiene il valore vero e proprio dell'attributo

Una richiesta Access-Request in particolare contiene tipicamente l'identità dichiarata dall'utente, il materiale necessario a verificarla, l'identità del NAS, la porta attraverso cui l'utente accede, il servizio richiesto...  
Il materiale di autenticazione dipende dal metodo specifico: con *PAP* si usa **User-Password**, con *CHAP* invece **CHAP-Password** che contiene la risposta CHAP e le informazioni necessarie a verificarla.

#### Come RADIUS protegge la password se si usa PAP
Con PAP l'utente presenta username e password.  
Si assume nel modello presentato che il collegamento tra utente e NAS sia protetto, mentre tra NAS e server RADIUS possano esserci attaccanti. 

Per evitare di inoltrare le password in chiaro tra NAS e server, RADIUS usa un meccanismo basato sempre su MD5. Consideriamo una pass lunga al massimo 16 byte.  
Per prima cosa il NAS aggiunge byte 0 fino ad ottenere il blocco $p1$ da 16 byte. Dopodiché calcola:
$$b_1 = MD5(S || A)$$
dove $S$ è il segreto condiviso e $A$ è il Request-Authenticator della richiesta.

MD5 produce un hash da 16 byte --> il NAS combina in XOR il blocco della password con questo valore:
$$c_1 = p_1 \oplus b_1$$
Il NAS inserisce $c_1$ nel campo User-Password della richiesta Access-Request.  
Allora poichè il server conosce $S$ e riceve $A$, può calcolare $b_1$ e recuperare $p_1 = c_1 \oplus b_1$: ci si ricollega quindi di nuovo all'utilizzo di una **maschera combinata tramite XOR**. 

Se invece la pass era più lunga di 16 byte --> si effettua sempre padding con byte 0 fino a una lunghezza multipla di 16 byte e suddivisa in blocchi $p_1, p_2, ..., p_l$.  
Allora per il primo blocco:
$$b_1 = MD5(S || A) \quad c_1 = p_1 \oplus b_1$$
Per i blocchi successivi:
$$b_i = MD5(S || c_{i-1}) \quad c_i = p_i \oplus b_i$$
In questo modo si utilizza una maschera diversa per ogni blocco (simile a quantio visto con CBC nei cifrari a blocchi).

**Le critiche da fare a RADIUS per questo meccanismo sono:**
- impiego di costruzione crittografica specifica del protocollo invece di un algoritmo standard studiato per la cifratura
- dipendenza da MD5
- riutilizzo dello stesso segreto $S$ per proteggere la password e autenticare le risposte del server
- la dipendenza dalla corretta generazione del Request-Authenticator $A$ 

Le ultime due criticità sono evidenti in quanto se due richieste usano lo stesso $S$ E lo stesso $A$ allora per il primo blocco si ottiene la stessa maschera $b_1 = MD5(S || A)$, e se l'attaccante se ne accorge:
$$c_1 \oplus c_1' = p_1 \oplus b_1 \oplus p_1' \oplus b_1 = p_1 \oplus p_1'$$
a quel punto se conosce una delle due password può ricavare l'altra.

Vedremo in dettaglio gli attacchi più avanti, qui basta capire che il solo fatto di usare un hash non garantisce che una costruzione sia un generatore sicuro di keystream e che le collisioni note di MD5 non dimostrano che ogni costruzione basata su MD5 sia violabile allo stesso modo.

Prima di proseguire con CHAP ricordiamo le possibili risposte del server a una richiesta Access-Request:
- Access-Accept --> accesso autorizzato, può contenere configurazioni come il servizio concesso, l'indirizzo IP e altri parametri
- Access-Reject --> accesso negato, può dipendere da credenziali non valide o richiesta non autorizzata anche se l'identità è stata verificata
- Access-Challenge --> la procedura deve proseguire. In tal caso il NAS raccoglie nuove info dall'utente e invia un nuovo Access-Request.

#### Come RADIUS protegge la password se si usa CHAP
Con CHAP ricordiamo si usa una challenge e una risposta calcolata tramite il segreto dell'utente. Indichiamo:
$$q = \text{challenge} \quad r = \text{risposta} \quad P_u = \text{password dell'utente}$$
La forma semplificata che usiamo per il ragionamento è $r = H(q, P_u)$, ma in realtà nel CHAP-MD5 standard interviene anche un identificatore CHAP $j$ che serve a distinguere più challenge in una stessa sessione (diverso dall'identifier del pacchetto RADIUS!) $r = MD5(j || P_u || q)$.

Procedura:
1. il NAS genera la challenge $q$, e la invia all'utente
2. l'utente calcola la risposta $r$ usando la propria password
3. il NAS inoltra al server username, challenge e risposta CHAP
4. il server recupera il materiale segreto dell'utente, ricalcola la risposta e confronta per poi inviare Access-Accept oppure Access-Reject

La cosa importante è che quindi **il NAS può generare la challenge senza conoscere la password**, ma non può ovviamente verificare la risposta se non dispone del materiale segreto necessario.  
Nel formato RADIUS l'attributo CHAP-Password quindi **non contiene la password dell'utente, ma l'identificatore CHAP e la risposta calcolata**.

Inoltre è essenziale distinguere i ruoli di $q$ e $A$ nel protocollo: la challenge CHAP $q$ permette al server di verificare la risposta dell'utente, mentre il Request-Authenticator $A$ serve a collegare la risposta RADIUS alla richiesta corrente del NAS.  

(NB. Challenge e risposta CHAP possono essere intercettate, ma la password non passa mai nel canale di comunicazione come vuole CHAP e inoltre ovviamente la risposta CHAP è un hash e quindi serve solo per verifica tramite confronto, non per recuperare la password dell'utente! Nun se po' fa'!)

##### Attacco a CHAP con RADIUS
Consideriamo il meccanismo descritto finora, dove **gli attributi dell'Access-Request non sono protetti da MAC** e consideriamo un attaccante capace di modificarli nel canale NAS-server.  
L'attaccante non conosce né la password di Lisa, né il segreto condiviso $S$. Ha però osservato una precedente autenticazione valida di Lisa.
1. **L'attaccante registra uno scambio CHAP valido**: durante l'accesso legittimo, Lisa ha ricevuto una challenge $q_0 = 1234$ e ha prodotto la risposta $r_0 = H(q_0, P_{Lisa}) = \alpha \beta \gamma \delta$. Allora l'attaccante conserva la tupla valida:
   $$(\text{Lisa}, j_0, q_0, r_0)$$
   che soddisfa r_0 = MD5(j_0 || P_{Lisa} || q_0)
2. **L'attaccante avvia un nuovo accesso dichiarandosi Lisa** --> il NAS genera una nuova challenge $q_1 = 5678$.  
L'attaccante non conosce la password quindi non può generare la risposta corretta, allora invia un valore arbitrario $r_{fake}$. Il NAS non può verificarla direttamente --> prepara la richiesta RADIUS contenente:
   $$Q = (I_1, A_1, \text{Lisa}, j_1, q_1, r_{fake}, \text{servizio richiesto})$$
   Il NAS conserva $A_1$ per verificare la successiva risposta RADIUS come visto all'inizio. 
3. Se il server ricevesse questa richiesta invariata, l'autenticazione fallirebbe.  
Ma l'attaccante **modifica la richiesta tra NAS e server**: intercetta $Q$ e sostituisce i dati CHAP con quelli osservati nella vecchia autenticazione valida:
   $$(j_1, q_1, r_{fake}) \to (j_0, q_0, r_0)$$
   Lasciando invariati l'Identifier RADIUS $I_1$ e il Request-Authenticator $A_1$. Il server riceve quindi:
   $$Q' = (I_1, A_1, \text{Lisa}, j_0, q_0, r_0, \text{servizio richiesto})$$
   che non è replay dell'intero pacchetto (non sarebbe valido perché l'Identifier RADIUS è diverso), ma **solo dei campi CHAP**. 
4. **Il server verifica i dati sostituiti e risponde positivamente**: infatti per il server la coppia challenge-risposta è valida: $r_0 = MD5(j_0 || P_{Lisa} || q_0)$, quindi invia Access-Accept al NAS autenticato con $I_1$, $A_1$ e $S$. 
5. **Il NAS accetta una risposta autentica alla richiesta alterata**: riceve l'Access-Accept e verifica il tag usando il nonce $A_1$ che aveva conservato, la verifica riesce.

In questo caso il nonce non impedisce l'attacco perché l'attaccante non ripresenta una vecchia risposta radius, ma ne modifica una nuova intercettata.  
Questo dimostra come in questo caso **la freshness della risposta RADIUS non prova l'integrità della richiesta Access-Request!**

**Perché non basta memorizzare la richiesta sul NAS**? Il NAS potrebbe conservare il fatto di aver inviato $(\text{Lisa}, j_1, q_1, r_{fake})$, ma l'Access-Accept dal server non restituisce necessariamente quei dati in una forma autenticata che permetta di confrontarli con quelli effettivamente verificati da parte del server.  
Il tag della risposta copre solo la risposta e $A_1$, non contiene automaticamente la challenge e la risposta CHAP dell'Access-Request, i.e. il NAS può essere sicuro che il server ha risposto positivamente, ma non che abbia ricevuto gli stessi attributi che lui gli aveva inviato. 

Per eliminare la possibilità dell'attacco ci sono due possibilità:
1. **Proteggere anche l'integrità della richiesta**, di modo che l'attaccante non possa sostituire gli attributi
2. **Legare crittograficamente la risposta del server al contenuto della richiesta**, di modo che il NAS possa verificare che il server abbia elaborato la richiesta originale.



### RADIUS Security Weaknesses e come ovviare ad esse
A conti fatti, RADIUS per come l'abbiamo descritto (protocollo classico) ha due problemi principali:
1. **non cifra l'intero pacchetto**: vari attributi come username, identificazione del NAS e info sulla stazione che accede possono essere leggibili anche da chi intercetta il traffico (proteggere la password non equivale a proteggere tutte le info trasmesse)
2. **autenticazione della risposta, ma non della richiesta Access-Request**

#### Message Authenticator
Per risolvere il secondo problema e quindi proteggere anche la richiesta, viene introdotto l'attributo **Message Authenticator**.  
La sua struttura è un Type pari a 80, Length 18 byte (uno per Type, uno per Length e 16 per Value) e come Value il tag di 16 byte. 

Questa volta il valore del tag viene calcolato tramite **HMAC-MD5**, con il segreto condiviso $S$ come chiave:
$$t_{MA} = HMAC-MD5_S(\text{pacchetto RADIUS})$$
Per un Access-Request, il pacchetto in particolare è:
$$\text{Code} || I || \text{Length} || A || \text{Attributes}$$
Durante il calcolo, i 16 byte del valore dell'attributo Message Authenticator vengono considerati come **tutti 0**, per evitare che il tag debba includere se stesso.

Il server ricalcola il valore e lo confronta con quello ricevuto: se l'attaccante modifica gli attributi CHAP --> il tag non corrisponde, richiesta scartata. Message-Authenticator quindi risolve l'attacco discusso prima secondo l'approccio 1: protezione dell'integrità della richiesta.

Un limite della specifica storica è che *Message-Authenticator era obbligatorio solo per messaggi che contenessero EAP-Message, mentre era di scelta arbitraria in altri metodi come CHAP.*  
Ciò rappresentava un limite in quanto **la disponibilità di una protezione non implica che sia effettivamente usata**.

### Accortezze su $S$
Il segreto $S$ è condiviso tra NAS e server RADIUS, e va distinto dalla password dell'utente. Due possibili problemi:
1. **Un segreto debole**
2. **Un segreto condiviso tra molti NAS**

Per il primo punto bisogna ricordarsi che $\text{Lunghezza segreto} \neq \text{Entropia}$: un segreto lungo può essere comunque prevedibile se deriva da una parola comune o da uno schema facile da riconoscere.  
In particolare se il segreto è configurato tramite interfaccia testuale --> gli amministratori tendono a scegliere qualcosa di memorizzabile --> una chiave che dovrebbe essere casuale diventa password umana, esposta ad attacchi di dizionario.

Per il secondo: assumiamo che tutti i NAS scelgano lo stesso $S$ --> se un dispositivo viene compromesso, effetto a cascata: una singola compromissione coinvolge tutte le comunicazioni.

(Approfondimento orale, non sulle slide) A livello di memoria chiaramente memorizzare un segreto per ogni NAS è costoso --> un'alternativa è **derivare i segreti a partire da un segreto principale e dall'identità del NAS**.  
Se $M = \text{master secret} e $ID_i = \text{identità del NAS i}$, allora l'idea è che $S_i = F(M, ID_i)$.

Concretamente si potrebbe usare ad esempio 
$$S_i = HMAC-SHA256_M(\text{"RADIUS NAS"} || ID_i)$$
Il procedimento è che il sistema centrale conserva $M$, per ogni NAS calcola il segreto $S_i$ e lo invia ad ogni NAS, che lo memorizza (*chiaramente ogni NAS non conosce il master secret $M$, altrimenti di nuovo uno compromesso deriva tutti gli altri segreti*).  
In questo modo il server non deve memorizzare tutti gli $S_i$ ma può ricalcolarli al momento adatto usando l'identità del dispositivo.  
GLi ID possono essere pubblici, la sicurezza dipende quindi da $M$ e dalla funzione di derivazione $F$.

#### Attacco del dizionario (quindi offline) ad $S$ 
(Riprende le slide)  
Vediamo prima un possibile attacco offline ad $S$ usando **richiesta e risposta**.  
Se un attaccante intercetta una richiesta e la relativa risposta, allora conosce: **$A$, dalla richiesta**; **Code, $I$, Length e Attributes, dalla risposta**; e il **tag $t$ sempre dalla risposta**, dove ricordiamo che nel protocollo originale:

<img src="imgs/radius_tag.png" width="400"/>

**L'unico elemento sconosciuto della formula è $S$.**  
Per ogni canditato $\hat{S}$, allora l'attaccante calcola:
$$
\hat{t} = MD5(\text{Code} || I || \text{Length} || A || \text{Attributes} || \hat{S})
$$ 
per poi confrontare $\hat{t} = t$. Se coincidono, allora $\hat{S} = S$ e l'attaccante ha trovato il segreto condiviso.

Tutto questo è un attacco **offline**: dopo aver raccolto i messaggi l'attaccante non deve inviare ogni tentativo al server, che quindi non può proteggersi imponendo limiti, ritardi e blocchi...  
Se $S$ è una password umana debole, allora un dizionario (i.e. provare un elenco di password plausibili, basandosi su parole comuni o password emerse da precedenti violazioni) può ridurre di molto il numero di tentativi.

In questo senso c'è anche un problema di costruzione già visto con SHA256: il tag è della forma $MD5(\text{parte nota} || S)$ per cui l'attaccante può precomputare lo stato di MD5 dopo aver processato la parte nota e riutilizzarla durante la ricerca, velocizzando ulteriormente l'attacco.

In realtà l'attacco offline può essere condotto **anche usando una sola richiesta**.  
Si ricorda infatti che la protezione per il primo blocco della password è $c = p \oplus MD5(S || A)$, con $p$ password completata a 16 byte. Allora l'attaccante può avviare un tentativo di accesso con una pass scelta da lui e intercettare il relativo Access-Request che il NAS produce. 

Quindi l'attaccante conosce la password padded $p$, perché l'ha scelta lui; il ciphertext $c$ dal pacchetto e anche il nonce $A$, sempre dal pacchetto.  
Quindi può calcolare:
$$c \oplus p = (p \oplus MD5(S || A)) \oplus p = MD5(S || A)$$
ottenendo così $b = MD5(S || A)$, per ogni candidato $\hat{S}$ può quindi verificare se $\hat{b} = MD5(\hat{S} || A)$ coincide con $b$.  

Ciò dimostra esplicitamente perché riutilizzare lo stesso segreto $S$ anche per proteggere la password è problematico: l'attaccante può sfruttare il meccanismo in questione per risalire ad $S$, che poi può usare per falsificare le risposte del server e ottenere accesso alla rete.

#### Attacco (online) alla password dell'utente
Qui il prof non ci si focalizza molto. L'idea di un attacco a $p$ è la seguente: ipotizziamo che l'attaccante abbia intercettato, per un nonce $A$, $b = MD5(S || A)$. 

Allora anche senza conoscere $S$ può provare a indovinare la password dell'utente con una password candidata $\hat{p}$, lunga al massimo un blocco, e calcolare $\hat{c} = \hat{p} \oplus b$.

A quel punto l'attaccante può inviare al server richieste con la password cifrata $\hat{c}$, il nonce $A$ intercettato ed il nome della vittima --> se il server accetta richieste **così costruite** (nel senso che non stiamo facendo un access-request necessariamente, anche solo un accesso a un servizio che richieda solo questi dati) e non impedisce il riutilizzo di $A$, l'attaccante può verificare online le password candidate. Se il server accetta la password apposto. 

#### Un generatore pseudocasuale può essere inadatto alla sicurezza
RADIUS abbiamo visto usare un Request-Authenticator $A$ da 16 byte (128 bit) come nonce. Ma avere un campo di 128 bit non significa contenga 128 bit di imprevedibilità, si distinguono in questo senso tre aspetti:
1. **Periodo**: dopo quanti passi si ripete lo stato/sequenza del generatore
2. **Prevedibilità**: osservando alcuni output, si possono prevedere i successivi?
3. **Ripetizione degli output**: se lo stesso valore può comparire più volte anche prima della ripetizione dell'intero ciclo

Un periodo lungo non garantisce sicurezza crittografica, in quanto non elimina necessariamente la prevedibilità o la ripetizione degli output.

Per la generazione di valori di sicurezza serve un **CSPRNG** (Cryptographically Secure Pseudo-Random Number Generator), correttamente inizializzato con entropia. 

Ad esempio un generatore lineare congruenziale (LCG) del tipo:
$$x_{n+1} = (a x_n + c) \mod m$$
con $x_0$ seme iniziale produce una sequenza di numeri che sembrano irregolari, ma se l'attaccante conosce lo stato $x_t$ e i parametri allora può prevedere tutti i valori successivi. Lo stesso vale per operazioni come rand() in C, che spesso implementano proprio.

##### Espandere la lunghezza dell'output non crea nuova entropia
Supponiamo di avere un generatore debole con stato di 32 bit e periodo massimo $L = 2^{32}$ (debole perché birthday attack per collisione, nel caso il generatore non percorra distintamente tutti i valori prima di completare il ciclo, richiede in media $2^{16}$ operazioni). Se vogliamo generare un nonce da 128 bit allora avremmo tre possibilità:
1. **Ripetere lo stesso valore quattro volte**:
    $$A = x_t || x_t || x_t || x_t$$
    il campo occupa effettivamente 128 bit, ma l'entropia è sempre di 32 bit. Il pattern ripetuto rende inoltre evidente la debolezza
2. **Concatenare quattro output consecutivi**:
    $$A = x_t || x_{t+1} || x_{t+2} || x_{t+3}$$
    sembra una soluzione migliore in quanto le quattro parti possono essere diverse. Tuttavia essendo valori estratti da output successivi, non sono indipendenti: il primo valore determina gli altri tre per un attaccante che conosca il generatore --> entropia effettiva sempre 32 bit.  
    Inoltre, consumando come nell'esempio quattro valori consecutivi, il periodo effettivo del generatore torna allo stato iniziale dopo $\frac{2^{32}}{4} = 2^{30}$ autenticazioni.
3. **Applicare hash al valore di 32 bit**:
    $$A = H(x_t)$$
    questo approccio nasconde il pattern ripetuto, ma non aumenta lo spazio degli input: ci sono ancora al più $2^{32}$ risultati ottenibili.  L'attaccante può quindi enumerare gli input e calcolarne gli hash

Quindi **espandere la lunghezza dell'output non crea nuova entropia**: questa è una soluzione possibile come maschera estetica della debolezza del generatore sottostante, ma non è una soluzione crittografica.

#### Conseguenze della ripetizione del Request Authenticator
Se il NAS ripete un nonce già usato, una vecchia risposta autentica associata a quel nonce può diventare riutilizzabile se corrispondono anche gli altri campi. 

Immaginiamo che, con lo stesso segreto $S$ e lo stesso nonce $A$, si ripeta $b = MD5(S || A)$, quindi la maschera per il primo blocco della password è la stessa.

Allora per due primi blocchi di password si ha $c = p \oplus b$ e $c' = p' \oplus b$, quindi:
$$c \oplus c' = p \oplus b \oplus p' \oplus b = p \oplus p'$$
Di nuovo, ciò rivela una relazione utile per cui se l'attaccante conosce una delle due password (es. se ha inviato lui stesso una richiesta con una password scelta) può ricavare l'altra.

### Lessons learned from RADIUS
RADIUS aveva incorporato meccanismi propri per **autenticare le risposte** e **proteggere le password**. Ciò ha introdotto dipendenze da da MD5, gestione delicata dei nonce e altri aspetti da analizzare ed implementare correttamente.

Quindi abbiamo capito che in generale *un protocollo applicativo dovrebbe invece utilizzare protocolli di sicurezza standard, già studiati e consolidati per proteggere le comunicazioni* (es. TLS, IPSec, ...) invece di inventare una propria costruzione crittografica.  
Alcuni esempi sono **RADIUS over TLS (RADSEC)** e **RADIUS over IPsec**, dove l'applicazione RADIUS gestisce le proprie richieste e risposte, mentre un altro livello si occupa esclusivamente di proteggere il traffico. 

Si distingue in questo senso inoltre la *sicurezza della comunicazione* dalla *sicurezza dell'implementazione*: usare TLS o IPSec non risolve un buffer overflow, un errore nei permessi o un bug di parsing dei pacchetti...

Altro problema del RADIUS originario è **scarsa flessibilità crittografica**: la dipendenza da MD5 ha fatto sì che sostituirlo non equivalesse a cambiare semplicemente la funzione hash...

### AAA Evolutions: Beyond RADIUS (meno importante per l'esame)
RADIUS era nato per scenari più semplici di quelli successivi. L'evoluzione tecnologia ha reso evidenti ulteriori limiti oltre a quello crittografico: **Scalabilità**, **Flessibilità** (supportare nuovi servizi/tecnologie) e **Interoperabilità** (coordinare implementazioni e server diversi...).

Per quel che riguarda l'Interoperabilità, in una rete reale possono servire più server ad esempio per continuare il servizio in caso di guasto, load balancing, conservare info importanti come quelle dell'accounting...

Per la scalabilità invece, il problema è evidente: consideriamo un NAS con 48 porte occupate a sessioni che durano in media 20 min = 1200 secondi.  
Allora in media si possono gestire $\frac{48}{1200} = 0.04$ richieste al secondo, quindi circa 1 richiesta ogni $\frac{1}{0.04} = 25$ secondi.  
Con 10k NAS si arriva a un totale di $10k * 0.04 = 400$ richieste al secondo. Se ogni richiesta occupa circa 1000 byte --> $400 * 1000 * 8 = 3.2$ Mbit/s di traffico solo per le richieste, senza considerare risposte, accounting ed altri costi.  
Il carico medio inoltre non descrive i picchi: dopo un guasto o un riavvio, molti utenti possono tentare di autenticarsi contemporaneamente.  
**Il punto è che il sistema AAA può diventare punto di congestione**.

Storicamente per gestire questi problemi ci sono stati due percorsi: **estendere RADIUS** oppure **progettare un nuovo protocollo**.

Nel primo caso si parla di **RADIUS Extensions**/**RADIUSext**, con l'obiettivo di migliorare RADIUS conservando la compatibilità con quanto già installato (es. RADIUS over TLS). Ciò è importante per ridurre i costi e la complessità in quanto si parla di protocolli già largamente distribuiti.

#### Diameter
Nel secondo caso si parla invece di un nuovo protocollo da zero: **Diameter**. 

Per capire l'idea, vediamo un'analogia con la programmazione a oggetti: una classe base deve offrire funzionalità comuni, mentre le classi derivate specializzano il comportamento.  
Analogamente, **Diameter distingue Protocollo Base dalle Applicazioni Diameter**. Mentre il protocollo base definisce gli elementi comuni di comunicazione e gestione, le applicazioni specificano messaggi, attributi e regole necessarie a determinati servizi (es. Mobile IPv4 application che supporta la mobilità IP, NAS application per la gestione di NAS etc...)

<img src="imgs/diameter.png" width="500"/>

##### La scelta del trasporto
Una parte molto importante in generale è la **scelta del trasporto**.  
Tra un NAS e un server possono infatti circolare molte richieste indipendenti, legate a utenti diversi. Sono necessarie due proprietà:
1. **Affidabilità**: gestire la perdita di pacchetti
2. **Indipendenza**: evitare che il ritardo di uno scambio blocchi inutilmente tutti gli altri

UDP era utilizzato in RADIUS classico e non offriva da solo ritrasmissione o consegna affidabile.  
TCP al contrario offre un flusso affidabile e ordinato, tuttavia aprire e chiudere una connessione TCP per ogni richiesta è costoso. L'idea per risolvere il problema è mantenere una **connessione persistente**.

Un problema di TCP è l'**head-of-line blocking**: TCP consegna un unico flusso di byte nell'ordine corretto.  
Supponiamo che sullo stesso flusso viaggino $M_1, M_2, M_3$ relativi ad utenti diversi --> se si perde un segmento contenente byte iniziali del flusso, allora i successivi possono essere già arrivati ma TCP aspetta quelli mancanti prima di consegnarli in ordine. Quindi anche un msg indipendente può subire ritardo.

Per risolvere ciò si usa **SCTP (Stream Control Transmission Protocol)**: l'idea è che messaggi indipendenti possano utilizzare stream differenti --> il problema di consegna in uno stream non impone ritardi agli altri.  
Un'altra proprietà di SCTP è la **multi-homing**: un host può avere più interfacce e indirizzi IP, quindi se una interfaccia cade il protocollo può continuare a funzionare usando un altro indirizzo.

Tornando a Diameter, visto quanto è interessante in termini di nuovo protocollo, perché abbiamo citato anche RADIUSext? Perché nella rete reale esistono dispositivi intermedi come NAT, firewall, sistemi di rilevamento delle intrusioni, apparati che ispezionano il traffico, e questi dispositivi supportano solo determinati protocolli, bloccando il traffico di chi non riconoscono --> difficile migrare da RADIUS a Diameter.

##### Funzionamento molto in generale
Si mostra Diameter per capire in generale come progettare un protocollo capace di evolvere e funzionare in una rete con più server intermediari. 

Un *watchdog* è un meccanismo con cui un nodo verifica se il peer è ancora raggiungibile e risponde. Diameter prevede messaggi dedicati a questo controllo, in particolare con un piccolo intervallo casuale (detto jitter) per evitare che molti nodi effettuino il controllo contemporaneamente.

Passiamo al formato dei messaggi. Con Radius solo un byte per type, quindi solo 256 valori rappresentabili. Il problema è che se si vogliono aggiungere nuovi attributi, il numero di valori disponibili può esaurirsi.

Diameter invece usa gli **AVP (Attribute-Value Pair)**, con campi molto più ampi: in particolare Code è a 32 bit, quindi teoricamente si possono definire oltre 4 miliardi di attributi diversi:

<img src="imgs/diameter_header.png" width="400"/>

Il flag V, Vendor-Specific, indica se l'attributo è standard o specifico di un vendor. Nasce a questo punto una domanda: *se il destinatario non conosce un nuovo attributo, deve rifiutare tutto il messaggio o continuare?*  

In questo senso entra in gioco il bit M: **Mandatory**. Si tratta di un flag che indica se comprendere l'AVP è necessario per elaborare correttamente il messaggio:
- $M = 0$ --> il destinatario può ignorare l'AVP e continuare a elaborare il messaggio
- $M = 1$ --> il destinatario deve rifiutare il messaggio se non comprende l'AVP, segnalare il problema e procedere secondo il protocollo

##### hop-by-hop e end-to-end
Vediamo un esempio per capire una rete AAA con intermediari. Immaginiamo che un utente di un'università straniera acceda attraverso la rete di Tor Vergata e la richiesta debba raggiungere il sistema della sua università. 

Consideriamo NAS --> intermediario --> server finale. Quando diciamo "associare la risposta alla richiesta" possiamo riferirci a due livelli:
- lo scambio tra nodi adiacenti --> **hop-by-hop identifier**, 32 bit, associa richiesta e risposta in un singolo collegamento tra peer
- la richiesta originale lungo tutto il percorso --> **end-to-end**, 32 bit, identifica la richiesta lungo tutto il percorso e contribuisce a riconoscere duplicati

Es. la conferma di ricezione sul collegamento wifi non equivale alla conferma che il server finale ha ricevuto la richiesta.

**Hop-by-hop id**: supponiamo che il NAS invii all'intermediario la richiesta con id $h_1$. L'intermediario può inoltrarla nel server usando un proprio id $h_2$, conservando però il fatto che $h_1$ è associato a $h_2$. Quando il server risponde con $h_2$, l'intermediario può risalire a $h_1$ e inoltrare la risposta al NAS.

**End-to-end id**: questo valore viene mantenuto lungo il percorso dagli agenti Diameter. Insieme all'identità del nodo che ha originato la richiesta, permette di riconoscere quelle duplicate. 

**Il cambiamento più importante apportato da Diameter è il fatto che possa essere considerata come una rete con più nodi intermedi, intermediari con ruoli distinti, instradamento di messaggi e differenza tra operazioni locali ed end-to-end.**

